# MobiML Transforms Demo

In [ ]:
import pickle
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta

import sys
sys.path.append("../src")

from mobiml.utils import convert_wgs_to_utm
from mobiml.datasets import PortoTaxis, AISDK, PreprocessedAISDK, MOVER_ID
from mobiml.transforms import ODAggregator, TrajectoryAggregator, TrajectoryCreator, AreaAggregator, DeltaDatasetCreator
from mobiml.preprocessing import StationaryClientExtractor

## ODAggregator 

Aggregate the PortoTaxis in a hex grid with hourly bins

In [ ]:
taxis = PortoTaxis('./data/train.csv', nrows=10000)

In [ ]:
od_h3 = ODAggregator(taxis).get_od_for_h3(7, freq='1h')
od_h3.head()

## StationaryClientExtractor

Extracting AIS records around specific locations


In [ ]:
antennas = ['Point (11.96524 57.70730)', 'Point (11.63979 57.71941)', 'Point (11.78460 57.57255)']
antenna_radius_meters = 25000 
epsg_code = convert_wgs_to_utm(11.96524, 57.70730)

def create_client_gdf(clients, client_radius_meters) -> gpd.GeoDataFrame:
    ids =  [{'client': i} for i in range(len(clients))]
    df = pd.DataFrame(ids)
    df['geometry'] = gpd.GeoSeries.from_wkt(clients)
    gdf = gpd.GeoDataFrame(df, geometry=df.geometry, crs=4326)
    gdf = gdf.to_crs(epsg_code)
    gdf['geometry'] = gdf.buffer(client_radius_meters)
    return gdf.to_crs(4326)

buffered_antennas = create_client_gdf(antennas, antenna_radius_meters)
buffered_antennas.explore()

In [ ]:
min_lon, min_lat, max_lon, max_lat = buffered_antennas.geometry.total_bounds
aisdk = AISDK('./data/aisdk_20180208_sample.zip', min_lon, min_lat, max_lon, max_lat)
aisdk.plot()

In [ ]:
print(f"{datetime.now()} Extracting client data ...")
client_data = StationaryClientExtractor(aisdk).extract(buffered_antennas)

In [ ]:
out_path = './data/ais-extracted-stationary.feather'
print(f"{datetime.now()} Writing output to {out_path}")
client_data.to_feather(out_path)

In [ ]:
aisdk = PreprocessedAISDK(out_path)
aisdk.df

## TrajectoryCreator & TrajectoryAggregator

Extract trips and compute features

In [ ]:
h3_resolution = 8

def create_vessel_list(gdf):
    print(f"{datetime.now()} Creating vessel list ...")
    return gdf.groupby(MOVER_ID)[['ship_type','Name']].agg(pd.Series.mode)


def pickle_vessels(vessels, out_path):
    print(f"{datetime.now()} Writing vessels to {out_path} ...")
    with open(out_path, 'wb') as out_file:
        pickle.dump(vessels, out_file)


def pickle_trajectories(trajs, out_path):
    print(f"{datetime.now()} Writing trajectories to {out_path} ...")
    with open(out_path, 'wb') as out_file:
        pickle.dump(trajs, out_file)
    return out_path

In [ ]:
print(f"{datetime.now()} Loading data from {out_path} ...")
gdf =  gpd.read_feather(out_path)
vessels = create_vessel_list(gdf)

In [ ]:
print(f"{datetime.now()} Extracting trips ...")
trajs = TrajectoryCreator(gdf).get_trajs(gap_duration=timedelta(minutes=60))  

In [ ]:
print(f"{datetime.now()} Computing trajectory features ...")
trajs = TrajectoryAggregator(trajs, vessels).aggregate_trajs(h3_resolution)   

In [ ]:
traj_path = 'data/ais-extracted-trajectories.pickle' 
pickle_trajectories(trajs, traj_path)
pickle_vessels(vessels, 'data/ais-extracted-vessels.pickle')

## AreaAggregator

Aggregate point density, average speed, and average movement direction per polygon area.

In [ ]:
area_stats = AreaAggregator(aisdk).aggregate(buffered_antennas)
area_stats[["client", "point_count", "point_density", "avg_speed", "avg_direction"]]

In [ ]:
area_stats.explore(column="avg_speed", legend=True)

### Spatiotemporal aggregation

Pass a `freq` string to aggregate by both area and time bin. Any pandas resampling frequency works (e.g. `"1h"`, `"1D"`, `"W"`, `"ME"`).

In [ ]:
area_stats_hourly = AreaAggregator(aisdk).aggregate(buffered_antennas, freq="1h")
area_stats_hourly[["client", "t", "point_count", "point_density", "avg_speed", "avg_direction"]]

In [ ]:
# Map one time bin — filter to a specific hour and explore
t = area_stats_hourly["t"].iloc[1]
area_stats_hourly[area_stats_hourly["t"] == t].explore(column="avg_speed", legend=True)

### Static plots per time bin

In [ ]:
import matplotlib.pyplot as plt
import math

timestamps = sorted(area_stats_hourly["t"].dropna().unique())
n = len(timestamps)
ncols = min(3, n)
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), constrained_layout=True)
axes = axes.flatten() if n > 1 else [axes]

vmin = area_stats_hourly["avg_speed"].min()
vmax = area_stats_hourly["avg_speed"].max()

for ax, t in zip(axes, timestamps):
    subset = area_stats_hourly[area_stats_hourly["t"] == t]
    subset.plot(
        column="avg_speed",
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        cmap="viridis",
        edgecolor="grey",
        linewidth=0.5,
        legend=False,
        missing_kwds={"color": "lightgrey"},
    )
    ax.set_title(str(t))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

for ax in axes[n:]:
    ax.set_visible(False)

sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(vmin=vmin, vmax=vmax))
fig.colorbar(sm, ax=axes[:n], label="Avg speed (knots)", shrink=0.6, pad=0.02)

plt.show()


## DeltaDatasetCreator

Create a dataset of per-segment deltas in x, y, speed, direction, and time — suitable for training vessel location forecasting models (e.g. Nautilus).

Each row represents one trajectory segment and contains: `dx_curr`, `dy_curr`, `dspeed_curr`, `dcourse_curr`, `dt_curr`, `dt_next`, `dx_next`, `dy_next`.

In [ ]:
out_path = './data/ais-extracted-stationary.feather'
aisdk_client0 = PreprocessedAISDK(out_path)
aisdk_client0.df = aisdk_client0.df[aisdk_client0.df["client"] == 0].copy()

delta_dataset = DeltaDatasetCreator(aisdk_client0).get_delta_dataset(njobs=4)
delta_dataset.head()

### Windowed dataset

Slice the delta dataset into fixed-length windows for sequence modelling.

In [ ]:
windowed_dataset = DeltaDatasetCreator(aisdk_client0).get_windowed_dataset(njobs=4)
windowed_dataset